# Bài 5 — Phát hiện gian lận (Mini Project)
`bai5_fraud_detection.ipynb`

---

Xây dựng pipeline phát hiện gian lận end-to-end trên kiến trúc Lambda.

**Mục tiêu:**
- Huấn luyện mô hình ML trên batch data
- Tích hợp mô hình vào streaming pipeline
- Đánh giá hiệu năng và trade-off

> ⚠️ Bài 1, 2, 3 phải đã hoàn thành.

## Checkpoint 1 — Huấn luyện mô hình

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

DATA_RAW    = os.environ.get("DATA_RAW_PATH",    "/data/raw")
DATA_OUTPUT = os.environ.get("DATA_OUTPUT_PATH", "/data/output")
MODEL_PATH  = f"{DATA_OUTPUT}/model"

spark = SparkSession.builder.appName("Bai5_FraudDetection").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df         = spark.read.parquet(f"{DATA_RAW}/transactions_30days.parquet")
user_stats = spark.read.parquet(f"{DATA_OUTPUT}/batch/user_stats")
print(f"Transactions: {df.count():,} | Users: {user_stats.count():,}")

In [ ]:
# Tách train/test theo thời gian — không dùng random split
# TODO: tìm min_date, tính split_date = min_date + 25 ngày
# TODO: train_df = 25 ngày đầu, test_df = 5 ngày cuối
from pyspark.sql.functions import to_date
from datetime import timedelta

split_date = ...  # TODO
train_df   = ...  # TODO
test_df    = ...  # TODO

print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")

### Xây dựng đặc trưng

Feature vector phải bao gồm: `amount`, `category` (encoded), và ít nhất 2 đặc trưng từ `user_stats`.

In [ ]:
def build_features(df, user_stats_df):
    # TODO: join với user_stats để lấy avg_amount_user, days_active
    # TODO: thêm cột label = is_fraud.cast(integer)
    # TODO: fillna cho các cột numeric
    pass


train_feat = build_features(train_df, user_stats)
test_feat  = build_features(test_df,  user_stats)
train_feat.groupBy("label").count().show()

### Huấn luyện Pipeline

Dùng `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → Model.
Chọn `LogisticRegression` hoặc `RandomForestClassifier`, xử lý mất cân bằng lớp.

In [ ]:
# TODO: tạo StringIndexer cho category
# TODO: tạo OneHotEncoder
# TODO: tạo VectorAssembler gộp amount, category_vec, avg_amount_user, days_active
# TODO: tạo model (LR hoặc RF)
# TODO: tạo Pipeline và fit trên train_feat

trained_model = ...  # TODO
print("✓ Huấn luyện xong")

In [ ]:
# Đánh giá
predictions = trained_model.transform(test_feat)
mc_eval   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
precision = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedPrecision"})
recall    = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedRecall"})
f1        = mc_eval.evaluate(predictions, {mc_eval.metricName: "f1"})
auc       = BinaryClassificationEvaluator(labelCol="label").evaluate(predictions)

print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
predictions.groupBy("label","prediction").count().orderBy("label","prediction").show()

# Lưu mô hình
trained_model.write().overwrite().save(MODEL_PATH)
print(f"✓ Đã lưu vào {MODEL_PATH}")

## Checkpoint 2 — Streaming inference

> ⚠️ Đảm bảo producer từ Bài 1 đang chạy.

In [ ]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import PipelineModel

DATA_OUTPUT     = os.environ.get("DATA_OUTPUT_PATH", "/data/output")
KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:9093")
MODEL_PATH      = f"{DATA_OUTPUT}/model"
PRED_PATH       = f"{DATA_OUTPUT}/fraud_predictions"
CKPT_PATH       = f"{DATA_OUTPUT}/checkpoints/fraud"

spark = SparkSession.builder.appName("Bai5_Streaming").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

trained_model = PipelineModel.load(MODEL_PATH)
user_stats    = spark.read.parquet(f"{DATA_OUTPUT}/batch/user_stats")
print("✓ Model và user_stats đã load")

In [ ]:
TRANSACTION_SCHEMA = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id",        StringType()),
    StructField("product_id",     StringType()),
    StructField("amount",         DoubleType()),
    StructField("category",       StringType()),
    StructField("timestamp",      StringType()),
    StructField("is_fraud",       BooleanType()),
])

stream_raw = spark.readStream.format("kafka")     .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)     .option("subscribe", "transactions")     .option("startingOffsets", "latest")     .load()     .select(F.from_json(F.col("value").cast("string"), TRANSACTION_SCHEMA).alias("d"))     .select("d.*")

In [ ]:
def process_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return
    # TODO: join với user_stats (dùng broadcast join)
    # TODO: fillna cho avg_amount_user và days_active
    # TODO: áp dụng trained_model.transform()
    # TODO: select output gồm: transaction_id, user_id, amount, category,
    #        predicted_label, prediction_probability, processing_time
    # TODO: ghi ra PRED_PATH dạng Parquet mode="append"
    pass


query = stream_raw.writeStream     .foreachBatch(process_batch)     .option("checkpointLocation", CKPT_PATH)     .trigger(processingTime="15 seconds")     .start()

print("✓ Pipeline chạy — dừng bằng query.stop()")

In [ ]:
import time
print("Chạy 180 giây...")
time.sleep(180)
query.stop()
print("✓ Đã dừng")

## Checkpoint 3 — Đánh giá và báo cáo

In [ ]:
pred_df = spark.read.parquet(PRED_PATH)
total   = pred_df.count()
print(f"Tổng giao dịch đã xử lý: {total:,}")

# TODO: tính tỷ lệ fraud dự đoán
# TODO: phân bổ prediction_probability (dùng percentile_approx)
# TODO: tính độ trễ trung bình (processing_time - timestamp)
avg_latency = ...  # TODO
print(f"Độ trễ trung bình: {avg_latency:.1f} giây")
print("✓ Đạt < 5 giây" if avg_latency < 5 else "✗ Chưa đạt — xem lại trigger interval")

In [ ]:
# So sánh ML vs rule-based (amount > 3000)
comparison  = pred_df.withColumn("rule_flag", F.col("amount") > 3000)
ml_fraud    = comparison.filter(F.col("predicted_label") == 1.0).count()
rule_fraud  = comparison.filter(F.col("rule_flag") == True).count()
both        = comparison.filter((F.col("predicted_label")==1.0) & F.col("rule_flag")).count()

print(f"Tổng         : {total:,}")
print(f"ML fraud     : {ml_fraud:,} ({ml_fraud/total:.2%})")
print(f"Rule fraud   : {rule_fraud:,} ({rule_fraud/total:.2%})")
print(f"Cả hai       : {both:,}")

### Nhận xét

*(Viết tại đây — khoảng 200–300 từ)*

**Gợi ý:**
1. Điểm mạnh của hệ thống
2. Hạn chế hiện tại
3. Ít nhất 2 cải tiến cụ thể
4. ML vs rule-based: khi nào dùng cái nào?

## Kiểm tra đầu ra

In [ ]:
import os
print("=" * 45)
print("KIỂM TRA ĐẦU RA BÀI 5")
print("=" * 45)

checks = [
    ("Model đã lưu vào /data/output/model/",
        os.path.exists(MODEL_PATH) and len(os.listdir(MODEL_PATH)) > 0),
    (f"Precision ≥ 0.65 (hiện tại: {precision:.3f})", precision >= 0.65),
    ("fraud_predictions tồn tại",
        os.path.exists(PRED_PATH) and
        any(f.endswith(".parquet") for _,_,fs in os.walk(PRED_PATH) for f in fs)),
]
try:
    col_ok = all(c in pred_df.columns for c in
                 ["transaction_id","user_id","amount","predicted_label",
                  "prediction_probability","processing_time"])
    checks.append(("Đủ 6 cột đầu ra", col_ok))
except: checks.append(("Đủ 6 cột đầu ra", False))

try: checks.append((f"Độ trễ TB < 5 giây ({avg_latency:.1f}s)", avg_latency < 5))
except: checks.append(("Độ trễ TB < 5 giây", False))

all_passed = True
for name, passed in checks:
    print(f"  [{'✓' if passed else '✗'}] {name}")
    if not passed: all_passed = False
print()
print("✓ Bài 5 hoàn thành!" if all_passed else "✗ Xem lại checkpoint tương ứng.")